In [ ]:
import anthropic
import os
from dotenv import load_dotenv

load_dotenv()

apikey = os.getenv("Claude_API_KEY")

client = anthropic.Anthropic(api_key=apikey)

for model in client.models.list():
    print(model.id)

In [ ]:
import os
from dotenv import load_dotenv
import anthropic

load_dotenv()

apikey = os.getenv("Claude_API_KEY")

client = anthropic.Anthropic(api_key=apikey)

# No temperature: this model doesn't accept sampling parameters
response = client.beta.messages.create(
    model="claude-opus-5-5",
    max_tokens=16000,
    betas=["server-side-fallback-2026-07-01"],
    fallbacks="default",  # if Claude declines, the API retries on a fallback model
    messages=[{"role": "user", "content": "write a python code for reverse string"}],
)

for block in response.content:
    if block.type == "text":
        print(block.text)

In [ ]:
#Fixed Chunking Strategy

from langchain_text_splitters import RecursiveCharacterTextSplitter

GCPData = """Google Cloud offers the most integrated AI stack for Agentic deployments — Gemini LLMs, Vertex AI Agents, managed vector search, and a global serverless infrastructure. All key agent primitives are available as managed services.
Google pioneered the Transformer architecture and has the deepest integration between AI models (Gemini) and infrastructure. Vertex AI Agent Builder lets you deploy production agents without managing servers, with built-in grounding, RAG, and evaluation.
Vertex AI Agent Builder is Google's fully-managed service for deploying production-grade agents. It handles hosting, scaling, session management, grounding, and RAG — so you focus on business logic.
"""

print(GCPData.split(".")[0])

# splitter = RecursiveCharacterTextSplitter(chunk_size=70, chunk_overlap=20, separator=" ")
# sp = splitter.split_text(GCPData)
# for i, chunk in enumerate(sp):
#     print(f"Chunk {i+1}: {chunk}")

splitter = RecursiveCharacterTextSplitter(chunk_size=70, chunk_overlap=20, separator=" ")
sp = splitter.split_text(GCPData)
for i, chunk in enumerate(sp):
    print(f"Chunk {i+1}: {chunk}")

Google Cloud offers the most integrated AI stack for Agentic deployments — Gemini LLMs, Vertex AI Agents, managed vector search, and a global serverless infrastructure


In [15]:
#2nd Chunking Strategy: Recursive Character Text Splitter
from langchain_text_splitters import RecursiveCharacterTextSplitter

text = """
Python Lists
Lists are mutable collections.
You can add, remove, or modify list elements.

Python Tuples
Tuples are immutable collections.
Their elements cannot be changed after creation.

Python Dictionaries
Dictionaries store information as key-value pairs.
They are useful for looking up values using keys.
"""

splitter = RecursiveCharacterTextSplitter(
    chunk_size=50,
    chunk_overlap=10,
    length_function=len,
    separators=["\n\n", "\n", " ", ""]
)
#\n\n -- 

chunks = splitter.split_text(text)

for i, chunk in enumerate(chunks, start=1):
    print(f"\n--- Chunk {i} ---")
    print(chunk)
    print("Length:", len(chunk))




--- Chunk 1 ---
Python Lists
Lists are mutable collections.
Length: 43

--- Chunk 2 ---
You can add, remove, or modify list elements.
Length: 45

--- Chunk 3 ---
Python Tuples
Tuples are immutable collections.
Length: 47

--- Chunk 4 ---
Their elements cannot be changed after creation.
Length: 48

--- Chunk 5 ---
Python Dictionaries
Length: 19

--- Chunk 6 ---
Dictionaries store information as key-value
Length: 43

--- Chunk 7 ---
key-value pairs.
Length: 16

--- Chunk 8 ---
They are useful for looking up values using keys.
Length: 49


In [20]:
#Read PDF File 
from PyPDF2 import PdfReader

reader = PdfReader("employee_leave_policy.pdf")
text = ""
for page in reader.pages:
    text += page.extract_text() + "\n"

print(text)

Employee Leave Policy  Sample document
Page 1
EMPLOYEE LEAVE POLICY
 Sample Company Policy Document
 
Policy ID
HR-LEAVE-001
Version
1.0 — Sample
Effective date
1 January 2026 (illustrative)
Policy owner
Human Resources
Review cycle
Annually or when applicable law changes
Purpose
This document describes a sample framework for requesting, approving, recording, and administering employee
leave. It is designed as a realistic example for document-processing and Retrieval-Augmented Generation (RAG)
demonstrations.
Important notice
This is a fictional, illustrative policy and is not legal advice or an official company policy. Leave entitlements and
statutory requirements vary by country, state, employment category, and applicable law. An organization should
review and customize this document with HR and legal counsel before adopting it.
Document contents
1. Purpose, scope and principles
2. Leave types and illustrative entitlements
3. Request, approval and recording process
4. Special situat

In [ ]:
#2nd Chunking Strategy: Recursive Character Text Splitter
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=50,
    chunk_overlap=10,
    length_function=len,
    separators=["\n\n", "\n", " ", ""]
)
#\n\n -- 

chunks = splitter.split_text(text)

for i, chunk in enumerate(chunks, start=1):
    print(f"\n--- Chunk {i} ---")
    print(chunk)
    print("Length:", len(chunk))



#chunk1 



--- Chunk 1 ---
Employee Leave Policy  Sample document
Page 1
Length: 46

--- Chunk 2 ---
Page 1
EMPLOYEE LEAVE POLICY
Length: 28

--- Chunk 3 ---
Sample Company Policy Document
 
Policy ID
Length: 42

--- Chunk 4 ---
Policy ID
HR-LEAVE-001
Version
1.0 — Sample
Length: 43

--- Chunk 5 ---
Effective date
1 January 2026 (illustrative)
Length: 44

--- Chunk 6 ---
Policy owner
Human Resources
Review cycle
Length: 41

--- Chunk 7 ---
Annually or when applicable law changes
Purpose
Length: 47

--- Chunk 8 ---
This document describes a sample framework for
Length: 46

--- Chunk 9 ---
for requesting, approving, recording, and
Length: 41

--- Chunk 10 ---
and administering employee
Length: 26

--- Chunk 11 ---
leave. It is designed as a realistic example for
Length: 48

--- Chunk 12 ---
for document-processing and Retrieval-Augmented
Length: 47

--- Chunk 13 ---
Generation (RAG)
Length: 16

--- Chunk 14 ---
demonstrations.
Important notice
Length: 32

--- Chunk 15 ---
This is a fictional, ill

Semantic Chunking

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_experimental.text_splitter import SemanticChunker
import os
from dotenv import load_dotenv
from langchain_community.embeddings import HuggingFaceEmbeddings

load_dotenv()

# Claude has no embeddings API, so use the same local model as the RAG notebooks
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
print(embeddings)

# Load PDF
loader = PyPDFLoader("employee_leave_policy.pdf")
pages = loader.load()

# Combine extracted text
text = "\n\n".join(page.page_content for page in pages)

# Create semantic chunks
semantic_splitter = SemanticChunker(
    embeddings,
    breakpoint_threshold_type="percentile"
)

chunks = semantic_splitter.create_documents([text])

print("Total semantic chunks:", len(chunks))

for i, chunk in enumerate(chunks, start=1):
    print(f"\n--- Chunk {i} ---")
    print(chunk.page_content[:300])

In [2]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# Load an open-source embedding model
model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

# Example sentences
sentences = [
    "Employees receive 10 days of sick leave per year.",
    "Staff can take 10 days off when they are ill.",
    "Python is a programming language.",
    "Employees need manager approval for annual leave."
]

# Generate embeddings locally
vectors = model.encode(sentences)

print("Number of sentences:", len(sentences))
print("Embedding shape:", vectors.shape)

# Compare the meaning of the sentences
similarity_matrix = cosine_similarity(vectors)

print("\nCosine similarity matrix:")
print(similarity_matrix)

# Show the most similar pair for each sentence
for i, sentence in enumerate(sentences):
    scores = similarity_matrix[i].copy()
    scores[i] = -1  # Ignore comparison with itself

    best_match = scores.argmax()

    print(f"\nSentence: {sentence}")
    print(f"Most similar: {sentences[best_match]}")
    print(f"Similarity score: {scores[best_match]:.4f}")



c:\Users\Public\Documents\Python Training\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ImportError: `torch` can only be initialized once per process, so this module cannot be imported again: neither via `importlib.reload(torch)`, nor by re-importing after a failed `import torch`.